In [4]:
import pandas as pd
from pyprojroot import here

In [5]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [6]:
crime = pd.read_csv(RAW_DATA_DIR/"crime/Municipal-Delitos-2015-2025_abr2026.csv", encoding='latin-1')
population = pd.read_csv(INTERIM_DATA_DIR/"population_2015_2025B.csv")

In [7]:
crime['cvemun'] = crime['Cve. Municipio'] - crime['Clave_Ent'] * 1000
rows_to_drop = crime[crime['cvemun'] > 600].index
crime = crime.drop(index=rows_to_drop)

In [8]:
crime['CVEGEO'] = crime['Cve. Municipio'].astype(str).str.zfill(5)
population['CVEGEO'] = population['CVEGEO'].astype(str).str.zfill(5)

In [9]:
crime_2025 = crime[crime['Año'] == 2025].copy()
total_muns_2025 = crime_2025['CVEGEO'].unique().tolist()
len(total_muns_2025)

2478

In [10]:
crime = crime.rename(columns={'Año': 'year'})
crime['total'] = crime.iloc[:, 9:21].sum(axis=1)

In [11]:
A_crime = [ # Criminal activity associated with organized crime by the Federal Law Against Organized Crime
    'Secuestro',
    'Trata de personas',
    'Narcomenudeo',
    'Falsificación',
    'Corrupción de menores'
]

B_crime = [ # Criminal activity associated with organized crime by its general activity
    'Homicidio doloso',
    'Extorsión',
    'Evasión de presos'
]

C_crime = [ # Logistics related crime
    'Robo a transportista',
    'Robo de vehículo automotor'
]

D_crime = [
    'Delitos cometidos por servidores públicos',
    'Electorales'
]

In [12]:
A_df = crime[crime['Subtipo de delito'].isin(A_crime)].copy()
AB_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime)].copy()
ABC_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime)].copy()
ABCD_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime + D_crime)].copy()

In [13]:
grouped_AB = AB_df.groupby(['year', 'CVEGEO'])['total'].sum().drop(columns='Unnamed: 0').reset_index()

In [14]:
df = grouped_AB

munis_2025 = df[df['year'] == 2025][['CVEGEO']].drop_duplicates()

years = df['year'].unique()
skeleton = munis_2025.assign(key=1).merge(
    pd.DataFrame({'year': years, 'key': 1}), on='key'
).drop(columns='key')

df_fixed = skeleton.merge(df, on=['CVEGEO', 'year'], how='left')
df_fixed['total'] = df_fixed['total'].fillna(0).astype(int)
df_fixed = df_fixed.sort_values(['year', 'CVEGEO']).reset_index(drop=True)

In [15]:
result_AB = df_fixed.merge(population, on=['year', 'CVEGEO'], how='left')
result_AB.drop(columns='Unnamed: 0', inplace=True)
result_AB['crime_rate'] = result_AB['total'] / result_AB['population'] * 100000

In [16]:
result_AB.to_csv(INTERIM_DATA_DIR/"crime_rates_AB.csv", encoding='latin-1')

print(f" {len(result_AB)} municipalities processed")
print(result_AB.sample(10))

 27258 municipalities processed
      CVEGEO  year  total  population  crime_rate
2731   08046  2016     10     16264.6   61.483221
6189   20219  2017      0      5289.3    0.000000
26645  24035  2025   3526    728848.0  483.777139
22099  30174  2023     19    192587.2    9.865661
27178  31085  2025      0     35301.0    0.000000
19648  30201  2022      6     91302.8    6.571540
16586  21134  2021      6     42310.5   14.180877
17223  31042  2021      0     11131.5    0.000000
17203  31022  2021      0      8773.1    0.000000
24342  28042  2024      1     10143.4    9.858627
